# Notebook 01: Environment Setup

**Time:** 20 minutes  
**Prerequisites:** Notebook 00 complete  
**Goal:** Configure your learning path, initialize the LLM client, and preview Week 3 topics

This notebook will:
1. Set up Python path and imports
2. Choose your learning path (A/B/C)
3. Test your first API call
4. Preview the week's notebooks

In [5]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

# Force reload src modules
import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 01")

Setup complete -- ready for Notebook 01


---

## Part 1: Choose Your Path

| Path | Backend | Cost | Requirements |
|------|---------|------|-------------|
| **A** | Claude API (cloud) | ~$1-3 total | ANTHROPIC_API_KEY |
| **B** | Ollama (local) | Free | ~20GB RAM, ollama serve |
| **C** | Hybrid (both) | ~$0.50-1 | Both of the above |

In [6]:
# TODO 1: Choose your path and update MY_PATH
#
# Set to "A", "B", or "C" based on your setup.
# Then run this cell to save your choice.

MY_PATH = "A"  # <-- CHANGE THIS

# Save to config.py
config_path = os.path.join(parent_dir, 'src', 'config.py')
with open(config_path, 'r') as f:
    content = f.read()
content = content.replace(f'PATH = "{config.PATH}"', f'PATH = "{MY_PATH}"')
with open(config_path, 'w') as f:
    f.write(content)

importlib.reload(src.config)
import src.config as config

print(f"Path set to: {MY_PATH}")
print(f"Claude model: {config.CLAUDE_MODEL}")
print(f"Ollama model: {config.OLLAMA_MODEL}")

Path set to: A
Claude model: claude-sonnet-4-6
Ollama model: qwen3.5:27b


## Part 2: Initialize LLM Client

In [7]:
client  = LLMClient(path=config.PATH)
tracker = CostTracker()

print(f"\nClient ready (Path {config.PATH})")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001

Client ready (Path A)


## Part 3: Test API Call

In [8]:
print("=" * 65)
print("Test: First API Call")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="In 2-3 sentences, explain why data quality matters for LLM pretraining. Then briefly mention one modern tool (2025) for each: web scraping, OCR, and speech recognition.",
    system="You are a helpful ML engineering tutor.",
    max_tokens=300,
    temperature=0.7
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

Test: First API Call

Response in 7.2s
Model: claude-sonnet-4-6
Tokens: 62 in, 278 out
Stop reason: end_turn
## Why Data Quality Matters for LLM Pretraining

LLMs learn statistical patterns directly from their training data, so toxic, duplicated, or factually inconsistent content gets baked into model weights — leading to hallucinations, biased outputs, and poor generalization. High-quality, diverse data is arguably more impactful than model architecture choices, as demonstrated by models like Llama 3 and Mistral prioritizing aggressive data curation over simply scaling raw data volume.

---

## Modern Tools (2025)

| Domain | Tool | Why Notable |
|---|---|---|
| **Web Scraping** | **Crawl4AI** | Open-source, LLM-optimized crawler that outputs clean markdown, handles JavaScript rendering, and is designed specifically for AI dataset pipelines |
| **OCR** | **Docling** (IBM) | Handles complex PDFs, tables, and figures with high fidelity; widely adopted for converting documents into LLM-r

## Part 4: Path Rationale

In [9]:
# TODO 2: Explain why you chose your path

todo1_reflection = """
[YOUR REFLECTION HERE]

- Why did you choose Path A/B/C?
- What trade-offs did you consider (cost vs. speed vs. flexibility)?
- Have you used Ollama before? If so, how does it compare to cloud APIs?
"""

# Save path selection
path_doc = f"""# Week 3 Path Selection\n\n**Path:** {config.PATH}\n**Claude Model:** {config.CLAUDE_MODEL}\n**Ollama Model:** {config.OLLAMA_MODEL}\n\n## Rationale\n\n{todo1_reflection}\n"""

with open(os.path.join(outputs_dir, 'path_selection.md'), 'w') as f:
    f.write(path_doc)

print("Path selection saved to outputs/path_selection.md")
print(todo1_reflection)

Path selection saved to outputs/path_selection.md

[YOUR REFLECTION HERE]

- Why did you choose Path A/B/C?
- What trade-offs did you consider (cost vs. speed vs. flexibility)?
- Have you used Ollama before? If so, how does it compare to cloud APIs?



## Part 5: Week 3 Preview

| Notebook | Topic | Time | What You'll Build |
|----------|-------|------|------------------|
| 02 | Web Scraping & Text Extraction | 30 min | trafilatura vs Crawl4AI comparison |
| 03 | Document OCR & PDF Extraction | 30 min | Tesseract vs Marker vs Docling |
| 04 | Speech Recognition (ASR) | 25 min | faster-whisper transcription pipeline |
| 05 | Data Cleaning Pipeline | 30 min | MinHash dedup + PII removal + DataTrove |
| 06 | Text-to-Speech & Voice Synthesis | 25 min | Kokoro TTS + edge-tts |
| 07 | Voice Agent with Pipecat | 30 min | Real-time ASR -> LLM -> TTS pipeline |
| 08 | Project Integration | 35 min | Apply week's tools to capstone project |

In [10]:
# Save setup summary
import platform

summary = f"""Week 3 Setup Summary
==================
Date: {time.strftime('%Y-%m-%d %H:%M:%S')}
Python: {sys.version}
Platform: {platform.platform()}
Path: {config.PATH}
Claude Model: {config.CLAUDE_MODEL}
Ollama Model: {config.OLLAMA_MODEL}
API Test: {'PASS' if 'error' not in response else 'FAIL'}
"""

with open(os.path.join(outputs_dir, 'setup_summary.txt'), 'w') as f:
    f.write(summary)

print(summary)

Week 3 Setup Summary
Date: 2026-04-07 00:43:27
Python: 3.11.15 (main, Mar 25 2026, 02:58:47) [Clang 22.1.1 ]
Platform: macOS-26.3.1-arm64-arm-64bit
Path: A
Claude Model: claude-sonnet-4-6
Ollama Model: qwen3.5:27b
API Test: PASS



In [11]:
# Save reflection
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'

full_reflection = f"""
### Path Selection

{_todo1}
"""

reflection_file = append_to_reflection(
    notebook="01",
    section_title="Environment Setup",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  62
Total output tokens: 278
Total cost:          $0.0044

Last 1 calls:
  1. [00:43:18] sonnet -- 62in/278out -- $0.0044


## Notebook 01 Complete!

**What you accomplished:**
- Configured learning path (A/B/C)
- Initialized LLM client and cost tracker
- Tested first API call
- Previewed Week 3 topics

**Next:** Open **Notebook 02: Web Scraping & Text Extraction**